# ASTR 12600 — Real Kepler preprocessing experiment

This notebook runs the *frozen*, source-inspired comparison of baseline Box Least Squares (BLS) with the same BLS method after fixed near-gap masking. The stars, thresholds and injection distribution were fixed in `PREPROCESSING_ABLATION_FROZEN.json` before the first experiment. Source motivation: Hatzes (2020), Oshagh/Barros & Faria (2018), and Robnik et al. (2026).

**Scientific warning:** Null curves are inverted/rolled *proxies*, not real Kepler false-alarm truth. This is an exploratory classroom-scale experiment, **not** NASA-quality planet confirmation or a numerical reproduction of the published Robnik study.

## 1. Retrieve source code from the public GitHub research repository

This step requires an internet connection to GitHub and NASA MAST when later downloading light curves. No university credentials or API keys are required for these public sources.

In [ ]:
import os, subprocess
repo_url="https://github.com/DrewThomas09/ASTR_FInal_Project.git"
repo="/content/ASTR_FInal_Project"
if not os.path.isdir(repo+"/.git"):
    subprocess.run(["git","clone","--depth","1",repo_url,repo],check=True)
else:
    subprocess.run(["git","-C",repo,"pull","--ff-only"],check=True)
os.chdir(repo)
print("Repository:",os.getcwd())
print("Commit:",subprocess.check_output(["git","rev-parse","--short","HEAD"],text=True).strip())

## 2. Install BLS, FITS and plotting packages

Astropy supplies the established BLS algorithm; we are *not* introducing a new BLS discovery algorithm.

In [ ]:
%pip -q install "numpy>=1.26,<3" "pandas>=2,<4" "matplotlib>=3.7,<4" "astropy>=6,<9" "scipy>=1.11,<2" "requests>=2.31,<3" "pytest>=8,<10"

## 3. Verify engineering tests

Passing these tests shows that masking, injection and statistical accounting obey their coded assumptions; it is not evidence that the enhanced method detects planets more effectively.

In [ ]:
subprocess.run(["python","-m","pytest","-q","tests/test_kepler_preprocessing.py","tests/test_preprocessing_ablation.py"],check=True)

## 4. Execute frozen calibration and held-out comparisons

The four predeclared FITS files are downloaded from NASA MAST if not locally cached. If an archive source is inaccessible, the script records the failure. Results save to a reproducible directory. Some runs depend on external archive availability.

In [ ]:
subprocess.run(["python","-u","preprocessing_ablation.py",
                "--manifest","PREPROCESSING_ABLATION_FROZEN.json",
                "--out","results/preprocessing_ablation"],check=True)

## 5. Inspect the original results and limitations

Report paired improvements *and* losses, sample sizes, proxy-null flagged fractions, and exact recovery rates. Do not conflate a statistical score with a confirmed new planet.

In [ ]:
import json
from pathlib import Path
from IPython.display import display,Image
out=Path("results/preprocessing_ablation")
report=json.loads((out/"results_and_limits.json").read_text())
print("Thresholds:",json.dumps(report["thresholds"],indent=2))
print("Held-out proxy null flags:",json.dumps(report["null_flags_by_method"],indent=2))
print("Paired outcomes:",json.dumps(report["paired_outcomes"],indent=2))
print("Limitations:",json.dumps(report["limitations"],indent=2))
if (out/"recovery_comparison.png").exists():
    display(Image(filename=str(out/"recovery_comparison.png")))
import pandas as pd
display(pd.read_csv(out/"source_accountability.csv"))
display(pd.read_csv(out/"heldout_injection_cases.csv").head(12))

## 6. Download the complete evidence

Includes the exact input-manifest snapshot, thresholds, per-star SHA-256 provenance, every trial, errors, figures and full limitations.

In [ ]:
import shutil
archive=shutil.make_archive("/content/ASTR_Preprocessing_Experiment","zip",
                            root_dir="results",base_dir="preprocessing_ablation")
from google.colab import files
files.download(archive)